# 군집 분석 - KMeans, PCA, DBSCAN

군집(Clustering)은 **정답(target)을 사용하지 않고 데이터의 패턴이 비슷한 것끼리 묶는 비지도 학습**입니다.

이번 노트에서는 다음 흐름으로 학습합니다.

1. 와인 데이터로 KMeans 군집화
2. Silhouette Score / Adjusted Rand Score로 군집 평가
3. Iris 데이터로 KMeans 군집화
4. PCA를 이용한 2차원 시각화
5. DBSCAN을 이용한 밀도 기반 군집화
6. KMeans와 DBSCAN의 차이 비교


In [ ]:
from common.libraries import *

# 이번 학습에서 새로 사용하는 라이브러리
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris


## 1. KMeans

KMeans는 데이터를 **미리 정한 K개의 군집**으로 나누는 방법입니다.

동작 흐름은 다음과 같습니다.

1. K개의 중심점을 정합니다.
2. 각 데이터를 가장 가까운 중심점의 군집에 배정합니다.
3. 각 군집의 중심점을 다시 계산합니다.
4. 중심점이 거의 변하지 않을 때까지 반복합니다.

### 특징
- `n_clusters`로 군집 개수를 미리 지정해야 합니다.
- 거리 기반 모델이므로 특성의 단위 차이가 크면 스케일링이 중요합니다.
- 군집 번호 `0`, `1`, `2` 자체에는 의미가 없습니다.


## 2. 와인 데이터 준비

이미 배운 데이터 로드와 스케일링은 한 번에 준비합니다.

KMeans는 거리 기반 모델이므로 `StandardScaler`를 사용해 특성의 크기를 맞춥니다.


In [ ]:
wine = pd.read_csv("./data/08_wine.csv")

wine_X = wine[["alcohol", "sugar", "pH"]]
wine_y = wine["class"]

wine_scaler = StandardScaler()
wine_X_scaled = wine_scaler.fit_transform(wine_X)

print("원본 데이터:", wine_X.shape)
print("스케일링 데이터:", wine_X_scaled.shape)


## 3. 와인 데이터 KMeans 군집화

KMeans는 비지도 학습이므로 모델 학습에는 `y`를 사용하지 않습니다.

여기서는 와인 데이터의 실제 클래스가 2개이므로 학습 목적상 `n_clusters=2`로 지정합니다.


In [ ]:
wine_kmeans = KMeans(
    n_clusters=2,
    n_init=10,
    random_state=42
)

wine_cluster = wine_kmeans.fit_predict(wine_X_scaled)

print("군집 번호:", np.unique(wine_cluster))
print("군집별 개수:")
print(pd.Series(wine_cluster).value_counts().sort_index())


## 4. 군집 평가

### Silhouette Score
각 데이터가 **자신의 군집에는 얼마나 잘 모여 있고 다른 군집과는 얼마나 잘 분리되어 있는지** 평가합니다.

- 범위: `-1 ~ 1`
- 1에 가까울수록 군집 간 분리가 잘 된 상태
- 정답 레이블 없이도 계산 가능

### Adjusted Rand Score (ARI)
실제 정답 레이블이 있을 때 **군집 결과와 실제 클래스가 얼마나 비슷한 구조인지** 비교합니다.

- 1에 가까울수록 실제 클래스 구조와 잘 일치
- 군집 번호가 실제 클래스 번호와 반대로 붙어도 문제없이 비교 가능


In [ ]:
wine_silhouette = silhouette_score(wine_X_scaled, wine_cluster)
wine_ari = adjusted_rand_score(wine_y, wine_cluster)

print("Silhouette Score:", round(wine_silhouette, 4))
print("Adjusted Rand Score:", round(wine_ari, 4))


## 5. 와인 군집 결과 확인

원본 특성과 실제 클래스, KMeans가 만든 군집 번호를 하나의 DataFrame으로 묶어 확인합니다.

`cluster`는 KMeans가 만든 군집 번호이고 `class`는 실제 데이터에 있던 정답입니다.


In [ ]:
wine_result = wine_X.copy()
wine_result["class"] = wine_y.to_numpy()
wine_result["cluster"] = wine_cluster

display(wine_result.head())

sns.scatterplot(
    data=wine_result,
    x="alcohol",
    y="sugar",
    hue="cluster"
)
plt.title("Wine - KMeans Clustering")
plt.show()


# 6. Iris 데이터로 KMeans 다시 확인

이번에는 `Iris` 데이터를 사용합니다.

앞의 와인 데이터와 변수 이름이 섞이지 않도록 `iris_X`, `iris_y`처럼 별도의 변수명을 사용합니다.


In [ ]:
iris = load_iris()

iris_X = iris.data
iris_y = iris.target

iris_scaler = StandardScaler()
iris_X_scaled = iris_scaler.fit_transform(iris_X)

print("독립변수:", iris_X.shape)
print("정답:", iris_y.shape)


In [ ]:
iris_kmeans = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=42
)

iris_cluster = iris_kmeans.fit_predict(iris_X_scaled)

iris_silhouette = silhouette_score(iris_X_scaled, iris_cluster)
iris_ari = adjusted_rand_score(iris_y, iris_cluster)

print("Silhouette Score:", round(iris_silhouette, 4))
print("Adjusted Rand Score:", round(iris_ari, 4))


## 7. PCA를 이용한 군집 시각화

Iris 데이터는 특성이 4개라서 그대로는 2차원 그래프로 표현하기 어렵습니다.

`PCA`는 여러 특성의 정보를 최대한 유지하면서 차원을 줄이는 방법입니다.

여기서는 시각화를 위해 **4개의 특성을 2개의 주성분으로 축소**합니다.


In [ ]:
pca = PCA(n_components=2)
iris_pca = pca.fit_transform(iris_X_scaled)

print("변환 전:", iris_X_scaled.shape)
print("변환 후:", iris_pca.shape)

plt.scatter(
    iris_pca[:, 0],
    iris_pca[:, 1],
    c=iris_cluster
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Iris - KMeans Clustering")
plt.show()


# 8. DBSCAN

DBSCAN은 **데이터의 밀도**를 기준으로 군집을 만드는 모델입니다.

### KMeans와의 차이
- KMeans: 군집 개수를 먼저 정해야 함
- DBSCAN: 군집 개수를 미리 정하지 않음
- DBSCAN은 밀도가 낮은 데이터를 `-1`로 표시하여 노이즈(이상치)로 구분할 수 있음

### 주요 파라미터
- `eps`: 하나의 이웃으로 판단할 거리 범위
- `min_samples`: 핵심 포인트가 되기 위해 필요한 최소 데이터 수


In [ ]:
iris_dbscan = DBSCAN(
    eps=0.6,
    min_samples=5
)

iris_dbscan_cluster = iris_dbscan.fit_predict(iris_X_scaled)

print("군집 번호:", np.unique(iris_dbscan_cluster))
print("군집별 개수:")
print(pd.Series(iris_dbscan_cluster).value_counts().sort_index())


## 9. DBSCAN의 노이즈 확인

DBSCAN에서 군집 번호가 `-1`인 데이터는 어느 군집에도 포함되지 않은 노이즈입니다.

군집의 모양을 확인할 때는 전체 결과와 노이즈를 제외한 결과를 함께 볼 수 있습니다.


In [ ]:
noise_mask = iris_dbscan_cluster == -1

iris_X_cleaned = iris_X_scaled[~noise_mask]
iris_dbscan_cleaned = iris_dbscan_cluster[~noise_mask]
iris_y_cleaned = iris_y[~noise_mask]

print("전체 데이터:", iris_X_scaled.shape[0])
print("노이즈:", noise_mask.sum())
print("노이즈 제거 후:", iris_X_cleaned.shape[0])


In [ ]:
iris_pca_all = pca.transform(iris_X_scaled)
iris_pca_cleaned = pca.transform(iris_X_cleaned)

plt.scatter(
    iris_pca_all[:, 0],
    iris_pca_all[:, 1],
    c=iris_dbscan_cluster
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Iris - DBSCAN (노이즈 포함)")
plt.show()

plt.scatter(
    iris_pca_cleaned[:, 0],
    iris_pca_cleaned[:, 1],
    c=iris_dbscan_cleaned
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Iris - DBSCAN (노이즈 제거)")
plt.show()


## 10. DBSCAN 평가

Silhouette Score는 **최소 2개의 군집이 존재해야 계산할 수 있습니다.**

DBSCAN은 파라미터에 따라 하나의 군집만 만들어질 수도 있으므로 군집 개수를 확인한 뒤 평가합니다.

ARI는 실제 Iris 클래스와 DBSCAN 군집 결과의 구조적 유사도를 확인하기 위해 사용합니다.


In [ ]:
dbscan_cluster_count = len(set(iris_dbscan_cluster)) - (1 if -1 in iris_dbscan_cluster else 0)

if dbscan_cluster_count >= 2:
    dbscan_silhouette = silhouette_score(iris_X_scaled, iris_dbscan_cluster)
    print("Silhouette Score (노이즈 포함):", round(dbscan_silhouette, 4))
else:
    print("Silhouette Score: 군집이 2개 미만이라 계산하지 않습니다.")

dbscan_ari = adjusted_rand_score(iris_y, iris_dbscan_cluster)
print("Adjusted Rand Score:", round(dbscan_ari, 4))

cleaned_cluster_count = len(np.unique(iris_dbscan_cleaned))

if cleaned_cluster_count >= 2:
    cleaned_silhouette = silhouette_score(iris_X_cleaned, iris_dbscan_cleaned)
    print("Silhouette Score (노이즈 제거):", round(cleaned_silhouette, 4))
else:
    print("노이즈 제거 후에도 군집이 2개 미만이라 Silhouette Score를 계산하지 않습니다.")

cleaned_ari = adjusted_rand_score(iris_y_cleaned, iris_dbscan_cleaned)
print("Adjusted Rand Score (노이즈 제거):", round(cleaned_ari, 4))


# 정리

### KMeans
- 중심점과의 거리를 기준으로 데이터를 군집화
- `n_clusters`로 군집 개수를 미리 지정
- 거리 기반이므로 스케일링이 중요

### PCA
- 여러 특성을 더 적은 차원으로 축소
- 이번 학습에서는 군집 결과를 2차원으로 시각화하기 위해 사용

### DBSCAN
- 데이터의 밀도를 기준으로 군집화
- 군집 개수를 미리 정하지 않아도 됨
- 노이즈를 `-1`로 구분할 수 있음

### 군집 평가
- `silhouette_score` : 군집 내부 응집도와 군집 간 분리도 평가
- `adjusted_rand_score` : 실제 정답이 있을 때 군집 결과와 정답 구조의 유사도 평가
